In [1]:
from transformers import BertTokenizer , BertForQuestionAnswering
import torch

import warnings
warnings.filterwarnings("ignore")

In [2]:
model_name = "bert-large-uncased-whole-word-masking-finetuned-squad"

In [4]:
tokenizer = BertTokenizer.from_pretrained(model_name)

model = BertForQuestionAnswering.from_pretrained(model_name)

Loading weights: 100%|██████████| 391/391 [00:00<00:00, 11626.23it/s]
[transformers] BertForQuestionAnswering LOAD REPORT from: bert-large-uncased-whole-word-masking-finetuned-squad
Key                      | Status     |  | 
-------------------------+------------+--+-
bert.pooler.dense.weight | UNEXPECTED |  | 
bert.pooler.dense.bias   | UNEXPECTED |  | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.


In [14]:
def predict_answer(context, question):

    encoding = tokenizer(
        question,
        context,
        return_tensors="pt",
        max_length=512,
        truncation="only_second"
    )

    model.eval()

    with torch.no_grad():
        outputs = model(
            **encoding,
            return_dict=True
        )

    start_scores = outputs.start_logits
    end_scores = outputs.end_logits

    start_index = torch.argmax(start_scores, dim=1).item()
    end_index = torch.argmax(end_scores, dim=1).item()

    answer_ids = encoding["input_ids"][0][start_index:end_index + 1]

    answer = tokenizer.decode(
        answer_ids,
        skip_special_tokens=True
    )

    return answer

In [17]:
context = """
Artificial intelligence has become one of the most important technologies
in the modern world. It is widely used in healthcare, education,
transportation, finance, and many other industries.

In 2020, a research team at Stanford University started a project called
Smart Healthcare AI. The main goal of this project was to develop an
artificial intelligence system capable of detecting diseases at an early
stage.

The project was led by Dr. James Anderson, a computer scientist
specializing in machine learning and medical image processing.
The research team consisted of 25 scientists and engineers from
different countries.

During the first phase of the project, researchers collected more than
100,000 medical images from hospitals and research institutions.
These images were used to train deep learning models to identify
signs of different diseases.

In 2021, the research team developed a deep learning model that could
detect lung cancer from medical images with an accuracy of 94 percent.
The system was tested in several hospitals and demonstrated promising
results.

In 2022, researchers expanded the system to detect heart diseases
and neurological disorders. They also developed a mobile application
that allowed doctors to access the artificial intelligence system
remotely.

By 2023, the Smart Healthcare AI project had received international
recognition for its contributions to medical technology.
The research team planned to improve the accuracy of the system
and make it available to hospitals around the world.

Experts believe that artificial intelligence will significantly
improve healthcare services in the future by helping doctors
diagnose diseases faster and more accurately.
"""

question1 = "Who led the Smart Healthcare AI project?"

answer = predict_answer(context, question1)

print("Question:", question1)
print("Predicted Answer:", answer)

Question: Who led the Smart Healthcare AI project?
Predicted Answer: dr. james anderson


In [18]:
question2 = "What was the accuracy of the deep learning model in detecting lung cancer?"

answer2 = predict_answer(context, question2)

print("Question 2:", question2)
print("Predicted Answer:", answer2)

Question 2: What was the accuracy of the deep learning model in detecting lung cancer?
Predicted Answer: 94 percent


In [19]:
question = "What technological solution did the researchers develop in 2022 to enable doctors to access the AI system remotely?"

answer = predict_answer(context, question)

print("Question:", question)
print("Predicted Answer:", answer)

Question: What technological solution did the researchers develop in 2022 to enable doctors to access the AI system remotely?
Predicted Answer: mobile application
